In [1]:
# --- CELL 2: CÀO CHI TIẾT TỪ DANH SÁCH URLS ---
import re
import time
from selenium.webdriver.common.by import By

def safe_get_text(driver, xpath):
    try:
        return driver.find_element(By.XPATH, xpath).text.strip()
    except:
        return None

import re

def extract_phone(driver):
    phone_pattern = re.compile(r'(\+?\d[\d\s\-().]{7,})')

    # cách 1: chuẩn nhất (data-item-id)
    try:
        els = driver.find_elements(
            By.XPATH,
            "//button[contains(@data-item-id,'phone')]//div[contains(@class,'Io6YTe')]"
        )
        for el in els:
            text = el.text.strip()
            if phone_pattern.search(text):
                return text
    except:
        pass

    # cách 2: tel link
    try:
        els = driver.find_elements(By.XPATH, "//a[starts-with(@href,'tel:')]")
        for el in els:
            text = el.text.strip()
            if phone_pattern.search(text):
                return text
    except:
        pass

    # ❌ fallback NGUY HIỂM → chỉ lấy nếu match regex phone rõ ràng
    try:
        els = driver.find_elements(By.XPATH, "//div[contains(@class,'Io6YTe')]")
        for e in els:
            text = e.text.strip()

            # 🔥 CHẶN ADDRESS
            if any(keyword in text.lower() for keyword in ["street", "đường", "phường", "quận", "district"]):
                continue

            if phone_pattern.fullmatch(text.replace(" ", "")):
                return text

    except:
        pass

    return ""

def normalize_url(url):
    # nếu có hl=en → đổi thành hl=vi
    if "hl=en" in url:
        url = url.replace("hl=en", "hl=vi")
    elif "hl=" not in url:
        # nếu chưa có hl → thêm vào
        if "?" in url:
            url += "&hl=vi"
        else:
            url += "?hl=vi"

    return url


import unicodedata
import re

def normalize_key(text):
    # 1. đưa về dạng unicode chuẩn
    text = unicodedata.normalize('NFD', text)
    
    # 2. remove dấu
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    
    # 3. đ → d
    text = text.replace('đ', 'd').replace('Đ', 'D')
    
    # 4. lowercase + snake_case
    text = text.lower()
    text = re.sub(r'[^a-z0-9]+', '_', text)
    text = re.sub(r'_+', '_', text).strip('_')

    return text


def scrape_place(url):
    url = normalize_url(url)
    driver.get(url)
    try:
        # Chờ tên quán xuất hiện để xác nhận page đã load
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1.DUwDvf")))
        time.sleep(2) 
    except:
        return None



    data = {"url": url}
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.F7nice")))
    # --- 1. Thông tin cơ bản (Dựa trên Class tiêu chuẩn) ---
    data["name"] = safe_get_text(driver, "//h1[contains(@class, 'DUwDvf')]")

    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((
            By.XPATH,
            "//div[contains(@class,'F7nice')]//span[@aria-hidden='true']"
        ))
    )
    data["avg_rating"] = driver.find_element(
        By.XPATH,
        "//div[contains(@class,'F7nice')]//span[@aria-hidden='true']"
    ).text

    rev_label = driver.find_element(
        By.XPATH,
        "//div[contains(@class,'F7nice')]//span[@role='img' and contains(@aria-label,'đánh giá')]"
    ).get_attribute("aria-label")

    data["review_count"] = re.search(r'\d+', re.sub(r'[^\d]', '', rev_label)).group()
    # Loại hình (Bistro/Restaurant) - Thường nằm cạnh Rating
    data["category"] = safe_get_text( driver, "//button[contains(@jsaction, 'category')]")

    # Range giá (Tìm theo biểu tượng ₫)
    data["price_range"] = safe_get_text(driver, "//span[contains(text(), '₫') and not(contains(text(), 'người'))]")



    # --- 2. Lấy Địa chỉ (Dùng Xpath nhắm vào Aria-label) ---
    try:
        # Tìm bất kỳ nút nào có nhãn bắt đầu bằng "Address: "
        addr_el = driver.find_elements(By.XPATH, "//button[contains(@aria-label, 'Address: ')]//div[contains(@class, 'Io6YTe')]")
        if not addr_el: # Dự phòng nếu là giao diện Tiếng Việt
            addr_el = driver.find_elements(By.XPATH, "//button[contains(@aria-label, 'Địa chỉ: ')]//div[contains(@class, 'Io6YTe')]")
        
        data["address"] = addr_el[0].text.strip() if addr_el else ""
    except:
        data["address"] = ""



    # --- 3. Lấy Số điện thoại (Dùng data-item-id chuẩn) ---
    try:
        WebDriverWait(driver, 8).until(
            lambda d: (
                len(d.find_elements(By.XPATH, "//button[contains(@data-item-id,'phone')]")) > 0
                or len(d.find_elements(By.XPATH, "//a[starts-with(@href,'tel:')]")) > 0
            )
        )
    except:
        pass
    data["phone"] = extract_phone(driver)



    # --- 4. Lấy Giờ mở cửa (Opening Hours) ---
    data["opening_hours"] = {}

    try:
        # 🔥 tìm icon
        icon = driver.find_elements(By.XPATH, "//span[contains(@aria-label,'mở cửa')]")

        if icon:
            # 🔥 lấy đúng div clickable (cha của icon)
            clickable = icon[0].find_element(By.XPATH, "./ancestor::div[contains(@class,'MkV9')]")

            # scroll + click
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", clickable)
            time.sleep(0.5)
            driver.execute_script("arguments[0].click();", clickable)

            # 🔥 wait đúng table (IMPORTANT)
            WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.XPATH, "//div[contains(@class,'t39EBf')]//table"))
            )

            # 🔥 lấy từng dòng
            rows = driver.find_elements(
                By.XPATH,
                "//div[contains(@class,'t39EBf')]//table//tr"
            )
            print("Rows:", len(rows))
            for row in rows:
                try:
                    day = row.find_element(By.XPATH, ".//td[1]").text.strip()
                    hours = row.find_element(By.XPATH, ".//td[2]").text.strip().replace("\n", " ")

                    data["opening_hours"][day] = hours
                except:
                    continue

    except Exception:
        pass



    # --- 5. Lấy Price Per Person (Nếu có) ---
    # try:
    #     # Google thường ghi "₫100,000–200,000 per person"
    #     pp_el = driver.find_elements(By.XPATH, "//*[contains(text(), 'per person')]")
    #     if pp_el:
    #         data["price_per_person"] = pp_el[0].text.strip()
    #         # Tìm số lượt vote giá ngay cạnh đó (thường nằm trong class BfVpR)
    #         votes = driver.find_elements(By.CLASS_NAME, "BfVpR")
    #         data["price_votes"] = votes[0].text.strip() if votes else ""
    # except:
    #     pass
    


    # --- 6. LGBTQ+ (Check sự tồn tại của text) ---
    lgbtq_friendly = driver.find_elements(By.XPATH, "//*[contains(text(), 'LGBTQ+')]")
    data["lgbtq_friendly"] = "có" if lgbtq_friendly else "unknown"



    # --- 7. Popular Times (Tách biệt từng ngày chuẩn 100%) ---
    data["popular_times"] = {}

    try:
        # Scroll tới khu vực Popular Times (nếu có)
        pop_section = driver.find_elements(By.XPATH, "//div[contains(@aria-label, 'Giờ đông khách')]")
        if not pop_section:
            return data

        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", pop_section[0])
        time.sleep(1)

        # Lấy toàn bộ 7 ngày (7 panes)
        day_panes = driver.find_elements(By.CSS_SELECTOR, "div.C7xf8b > div")

        # Lấy tên các ngày (header phía trên)
        #day_names = driver.find_elements(By.CSS_SELECTOR, "div.C7xf8b button")

        weekdays = ["Chủ Nhật", "Thứ Hai", "Thứ Ba", "Thứ Tư", "Thứ Năm", "Thứ Sáu", "Thứ Bảy"]

        for i, pane in enumerate(day_panes):
            try:
                day_name = weekdays[i] if i < 7 else f"day_{i}"

                bars = pane.find_elements(By.CSS_SELECTOR, "div.dpoVLd")

                day_stats = []
                for b in bars:
                    label = b.get_attribute("aria-label")
                    if label:
                        clean_label = label.replace("\u202f", " ").strip()
                        day_stats.append(clean_label)

                if day_stats:
                    data["popular_times"][day_name] = day_stats

            except:
                continue

    except Exception:
        pass

    

    # --- 8. ABOUT (Giới thiệu) ---
    data["about"] = {}

    try:
        # 🔥 click tab "Giới thiệu"
        about_btn = driver.find_elements(
            By.XPATH,
            "//button[contains(@aria-label,'Giới thiệu') or contains(@aria-label,'About')]"
        )

        if about_btn:
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", about_btn[0])
            time.sleep(0.5)
            driver.execute_script("arguments[0].click();", about_btn[0])

            # 🔥 wait content load
            WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((
                    By.XPATH,
                    "//div[contains(@class,'m6QErb') and contains(@class,'XiKgde')]"
                ))
            )

            # 🔥 lấy tất cả section (child)
            sections = driver.find_elements(
                By.XPATH,
                "//div[contains(@class,'m6QErb') and contains(@class,'XiKgde')]/div"
            )

            for sec in sections:
                try:
                    # 🔥 title (ví dụ: "Nổi tiếng về")
                    title_el = sec.find_element(By.XPATH, ".//h2")
                    title = title_el.text.strip()

                    # normalize key
                    key = normalize_key(title)

                    # 🔥 lấy tất cả item trong ul/li
                    items = sec.find_elements(By.XPATH, ".//li//span[@aria-label]")

                    values = []
                    for it in items:
                        val = it.text.strip()
                        if val:
                            values.append(val)

                    if values:
                        data["about"][key] = values

                except:
                    continue

    except Exception:
        pass

    


    return data


In [2]:
# --- CELL 3: ĐỌC URL + SELENIUM + CRAWL (IMPROVED) ---

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import json
import os
import random

# --- CONFIG ---
BATCH_SIZE = 600
BREAK_TIME = 5
OUTPUT_FILE = "metadata_missing_friday.json"

# --- 1. Đọc URLs ---
with open("missing_friday.txt", "r", encoding="utf-8") as f:
    urls = [line.strip() for line in f if line.strip()]
    #urls = urls[:10]  # GIỚI HẠN 10 URL ĐẦU TIÊN CHO AN TOÀN
print(f"Tổng số URL: {len(urls)}")

# --- 2. Hàm init driver ---
def init_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    return webdriver.Chrome(options=options)

# Xóa file cũ
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

driver = None
wait = None
current_batch = 0

# --- 3. Crawl ---
for index, url in enumerate(urls):

    # 🔥 RESET DRIVER + WARM-UP
    if index == 0 or current_batch >= BATCH_SIZE:
        if driver:
            print(f"\n☕ Reset sau {BATCH_SIZE} URLs...")
            try: driver.quit()
            except: pass
            time.sleep(BREAK_TIME)

        print("🚀 Init driver...")
        driver = init_driver()
        wait = WebDriverWait(driver, 10)

        # 🔥 WARM-UP = URL đầu tiên của batch
        warmup_url = "https://www.google.com/maps/place/PH%E1%BB%9E+HI%E1%BB%80N+%7C+PH%E1%BB%9E+NGON+QU%E1%BA%ACN+5/data=!4m7!3m6!1s0x31752fb69dc45ddf:0xc9dea8c16b465c48!8m2!3d10.7579661!4d106.6714435!16s%2Fg%2F11t7ysr25z!19sChIJ313EnbYvdTERSFxGa8Go3sk?authuser=0&hl=vi&rclk=1"
        print(f"🔥 Warm-up: {warmup_url}")
        try:
            _ = scrape_place(warmup_url)
            print("   ✅ Warm-up xong")
        except:
            print("   ⚠️ Warm-up lỗi nhưng vẫn chạy")

        current_batch = 0

    # --- Crawl thật ---
    print(f"[{index+1}/{len(urls)}] {url}")

    try:
        data = scrape_place(url)
        data["url_source"] = url

        # --- Ghi file kiểu append ---
        file_exists = os.path.exists(OUTPUT_FILE)

        with open(OUTPUT_FILE, "a", encoding="utf-8") as f_out:
            if not file_exists:
                f_out.write("[\n")
            else:
                f_out.write(",\n")

            f_out.write(json.dumps(data, ensure_ascii=False, indent=2))

        print("   ✔️ Saved")

    except Exception as e:
        print(f"   ⚠️ Lỗi: {e}")

    current_batch += 1

    # 🔥 random sleep (QUAN TRỌNG)
    time.sleep(random.uniform(2, 4))

# --- 4. Đóng JSON ---
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "a", encoding="utf-8") as f_out:
        f_out.write("\n]")

# --- 5. Quit ---
try: driver.quit()
except: pass

print("\n✅ DONE")

Tổng số URL: 53
🚀 Init driver...
🔥 Warm-up: https://www.google.com/maps/place/PH%E1%BB%9E+HI%E1%BB%80N+%7C+PH%E1%BB%9E+NGON+QU%E1%BA%ACN+5/data=!4m7!3m6!1s0x31752fb69dc45ddf:0xc9dea8c16b465c48!8m2!3d10.7579661!4d106.6714435!16s%2Fg%2F11t7ysr25z!19sChIJ313EnbYvdTERSFxGa8Go3sk?authuser=0&hl=vi&rclk=1
   ⚠️ Warm-up lỗi nhưng vẫn chạy
[1/53] https://www.google.com/maps/place/Nem+Nuong/data=!4m7!3m6!1s0x31752fcaed1ff91b:0x87e52aa522239334!8m2!3d10.7525695!4d106.6739911!16s%2Fg%2F11hjxpllqq!19sChIJG_kf7covdTERNJMjIqUq5Yc?authuser=0&hl=vi&rclk=1
   ✔️ Saved
[2/53] https://www.google.com/maps/place/Rin+Sushi/data=!4m7!3m6!1s0x31752f50c2d8898b:0x6fa9f70c8b188314!8m2!3d10.758304!4d106.6826312!16s%2Fg%2F11h16nt6p4!19sChIJi4nYwlAvdTERFIMYiwz3qW8?authuser=0&hl=vi&rclk=1
Rows: 7
   ✔️ Saved
[3/53] https://www.google.com/maps/place/H%E1%BB%A7+ti%E1%BA%BFu+nam+vang+Ph%C6%B0%E1%BB%A3ng/data=!4m7!3m6!1s0x31752efb6ec40471:0x2b3726cafa6a6678!8m2!3d10.7534294!4d106.6690772!16s%2Fg%2F11c1tt2dcw!19sChIJcQTEb